# HR Assistant Agent — Ground Truth Evaluations

This notebook demonstrates evaluation of an agentic application with ground truth using Amazon Bedrock AgentCore Evaluations:

| Interface | When to use |
|---|---|
| **EvaluationClient** | You already have agent sessions in CloudWatch. Evaluate specific sessions against reference inputs. |
| **OnDemandEvaluationDatasetRunner** | You have a test dataset. You want to invoke the agent for every scenario and evaluate the results. |

### The HR Assistant Agent

We'll deploy an **HR Assistant** for Acme Corp — a Strands agent that helps employees with:
- PTO balance checks and time-off requests
- HR policy lookups (PTO, remote work, parental leave)
- Benefits information (health, dental, vision, 401k)
- Pay stub retrieval

### What You'll Learn
- How to use `EvaluationClient` to evaluate existing agent sessions logged in Amazon CloudWatch with ground-truth references
- How to use `OnDemandEvaluationDatasetRunner` to run automated dataset evaluations
- How to interpret evaluation results across built-in evaluators (Correctness, GoalSuccessRate, Trajectory)

### Tutorial Details

| Information | Details |
|---|---|
| Agent framework | Strands Agents |
| Runtime | Amazon Bedrock AgentCore Runtime |
| Evaluation SDK | `bedrock-agentcore` |
| AWS services | AgentCore Runtime, AgentCore Evaluations, CloudWatch Logs |

### Prerequisites
- Python 3.10+
- AWS credentials with permissions for AgentCore, Lambda, CloudWatch, ECR, IAM
- Docker running locally (for agent container image build)

## Step 1: Install Dependencies

In [1]:
!pip install -r requirements.txt -q

In [2]:
%pip install -r requirements.txt --no-cache-dir --force-reinstall

INFO: pip is looking at multiple versions of opentelemetry-instrumentation-boto to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-instrumentation-boto to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of opentelemetry-instrumentation to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-instrumentation to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the de

In [3]:
import IPython

IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

## Step 2: Configuration

Import libraries and configure your AWS session.

In [1]:
import boto3
import json
import time
import uuid
from datetime import timedelta
from boto3.session import Session
from IPython.display import display, Markdown
import os
import requests
from strands.tools import tool

region = "us-east-1" # Add AWS region here 
boto_session = Session(region_name=region)
REGION = boto_session.region_name

print(f"Region  : {REGION}")



Region  : us-east-1


### Create Code for the Agent
Create agents folder if it's not created.

In [8]:
%rm -rf ./agents >> /dev/null
%mkdir ./agents
%rm Dockerfile >> /dev/null

rm: cannot remove ‘Dockerfile’: No such file or directory


In [9]:
# Capture the environmental varaibles that were passed in from the lifecycle
BEDROCK_REGION = os.environ.get('BEDROCK_REGION')
ARC_POLICY_ARN = os.environ.get('ARC_POLICY_ARN')
GUARDRAIL_ID = os.environ.get('GUARDRAIL_ID')
GUARDRAIL_VERSION = os.environ.get('GUARDRAIL_VERSION')
KNOWLEDGE_BASE_ID = os.environ.get('KNOWLEDGE_BASE_ID')

In [10]:
BEDROCK_REGION
ARC_POLICY_ARN

'arn:aws:bedrock:us-east-1:161615149547:automated-reasoning-policy/6if243lxvpvp:1'

## Step 3: Deploy the HR Assistant Agent

We deploy the HR Assistant to **AgentCore Runtime** using the AWS SDK directly (no starter toolkit).
The deployment steps are:

1. **ECR authentication** — use `boto3` to get a temporary ECR token and run `docker login`
2. **Docker build** — build a container image from `hr_assistant_agent.py` and the local `Dockerfile`
3. **Docker push** — push the image to Amazon ECR
4. **CreateAgentRuntime / UpdateAgentRuntime** — register or update the agent endpoint via `bedrock-agentcore-control`

If a `.bedrock_agentcore.yaml` from a previous run is present, its ECR repository, IAM role, and
existing agent ID are reused so the cell is idempotent (re-running triggers an update, not a duplicate create).

In [28]:
%%writefile agents/claimvalidation-agent-noAtoA.py

"""
CLaims Validation — Strands agent deployed on Bedrock AgentCore Runtime.

Tools ( RAG and ARC Hooks):
"""
import os
import logging
import asyncio
import boto3
import json
import uvicorn
from datetime import datetime
from strands.models import BedrockModel
from strands.tools import tool
from strands.multiagent.a2a import A2AServer
from fastapi import FastAPI
from strands.hooks import HookProvider, HookRegistry, MessageAddedEvent, BeforeModelCallEvent, BeforeToolCallEvent
from pydantic import BaseModel
from botocore.config import Config as BotocoreConfig
from strands.telemetry import StrandsTelemetry
from findings_utils import extract_reasoning_findings
from strands_tools import retrieve
import re
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool

# Configure the root strands logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Add a handler to see the logs
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s", 
    handlers=[logging.StreamHandler()]
)

app = BedrockAgentCoreApp()

# Setup tracing - commented out for now as this adds a lot of trace output that really isn't interesting
# NOTE: To send the OTEL data to an ADOT collector, additional exporter needs to be used
StrandsTelemetry().setup_console_exporter()

# The default values as pulled fomr the environment.  These values
# are replaced after the file is written out.
BEDROCK_REGION = '<REPLACE_BEDROCK_REGION>'
ARC_POLICY_ARN = '<REPLACE_ARC_POLICY_ARN>'
GUARDRAIL_ID = '<REPLACE_GUARDRAIL_ID>'
GUARDRAIL_VERSION = '<REPLACE_GUARDRAIL_VERSION>'
KNOWLEDGE_BASE_ID = '<REPLACE_KNOWLEDGE_BASE_ID>'

# Initialize Bedrock client to verify connectivity
bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name=BEDROCK_REGION
)

print(f"✓ AWS Bedrock configured for region: {BEDROCK_REGION}")

# Setup the environment for the agent and tool
# Allow for the metadata to be retrieved on sources from the KB
os.environ['RETRIEVE_ENABLE_METADATA_DEFAULT'] = 'true'

# Store the Knowledge Base ID in the environment to allow the tool
# to interact with the KB
os.environ['KNOWLEDGE_BASE_ID'] = KNOWLEDGE_BASE_ID

# Define a notification hook to listen to events and then process the result and call
# Automated Reasoning attached via the Guardrail and report on the findings.  This
# can be used possibly re-write the output or add a flag on if the output is correct.
# Define a notification hook to listen to events and then process the result and call
# Automated Reasoning attached via the Guardrail and report on the findings.  This
# can be used possibly re-write the output or add a flag on if the output is correct.
class NotifyOnlyGuardrailsHook(HookProvider):
    
    def __init__(self, guardrail_id: str, guardrail_version: str, arc_policy_arn: str):
        self.guardrail_id = guardrail_id
        self.guardrail_version = guardrail_version
        self.arc_policy_arn = arc_policy_arn
        self.bedrock_client = boto3.client("bedrock-runtime")
        self.input = ''
        self.claim_valid = True
        self.findings = ''
        self.policy_definition = {}
        self.before_tool_event_flag = False
        self.before_model_event_flag = False

        if self.arc_policy_arn:
            try:
                bedrock_client = boto3.client('bedrock')
                response = bedrock_client.export_automated_reasoning_policy_version(policyArn=self.arc_policy_arn)
                self.policy_definition = response.get('policyDefinition', {})
            except Exception as e:
                print(f"Error getting policy definition: {str(e)}")
                raise

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeModelCallEvent, self.before_model_event)
        registry.add_callback(BeforeToolCallEvent, self.before_tool_event)
        registry.add_callback(MessageAddedEvent, self.message_added)

    def message_added(self, event: MessageAddedEvent) -> None:
        if self.before_tool_event_flag:
            # Since a tool was called, just ignore this message addition
            self.before_tool_event_flag = False
            return
        
        # Get the content
        content = "".join(block.get("text", "") for block in event.message.get("content", []))

        # Determine the source
        if event.message.get("role") == "user":
            # Store the input for later usage and allow the loop to continue to process
            self.input = content
            return

        if not content:
            return
            #do something 

        # Capture if this is the first time that findings will be created
        first_findings = (not self.findings)

        # Format a request to send to the guardrail
        content_to_validate = [
            {"text": {"text": self.input, "qualifiers": ["query"]}},
            {"text": {"text": content, "qualifiers": ["guard_content"]}}
        ]
        
        # Call the guardrail
        response = self.bedrock_client.apply_guardrail(
            guardrailIdentifier=self.guardrail_id,
            guardrailVersion=self.guardrail_version,
            source="OUTPUT",
            content=content_to_validate
        )

        # Determine if the output is correct
        self.findings = extract_reasoning_findings(response, self.policy_definition)
         
        assessments = response.get("assessments", [])
        if assessments and len(assessments):
            self.claim_valid = False

        # Add information to the output
        if self.findings and first_findings:
            new_output = content
            new_output = new_output + f"\n*** FINDINGS: ***:\n{self.findings}"
            new_output = new_output + f"\n*** CLAIM VALID: ***:\n{self.claim_valid}"
            event.message["content"][0]["text"] = new_output
        
    def before_model_event(self, event: BeforeModelCallEvent) -> None:
        self.before_model_event_flag = True

    def before_tool_event(self, event: BeforeToolCallEvent) -> None:
        self.before_tool_event_flag = True

# Create structured output
class StructuredOutputModel(BaseModel):
    claim_valid: bool
    content: str
    findings: str

agent_instructions="""You are an expert automotive claims validaiton specialist that determines if the users auto insurance claim is valid based on the provided information and details within the policy contract.
    
You will be provided with JSON data that has claim information and vehicle damage information, you should:
1. Extract from the JSON data required claim information to be validated
2. Focus on time of event and time of claim creation
3. Focus on claims and coverage inconsistencies

Your responses should :
- If your response is "Valid claim", then output the provide the full JSON structure provided from the input
- If you response is "Invalid claim", then output the response "This Claim is Invalid and no appraisal nor settlement is required"
- If you response is "Invalid claim", then provide clear explanation on why is invalid in the output
- In cases where a clear outcome is not present, recommend the user to check with their insurance agent directly. 

Create a section called "Claims Policy Validation Explainability". Use the Automated Reasoning Checks rules to provide explanation and list the rules that were valid and not valid
Provide the following for this section:
he Automated Reasoning Checks rules to provide explanation and list the rules that were valid and not validz
## Finding 
**Finding Type:** 

### Translation:
#### Premises:
#### Claims:
#### Untranslated Claims:

**Confidence Score:** 

### Claims True Scenario:


Take your time to think though the answer and evalute carefully."

"""

# Create agent with the guardrail-protected model
myagent = Agent(
    name="Claims Validator - ARC",
    description="A Single agent with ARC Claims Validation tools capabilities",
    hooks=[NotifyOnlyGuardrailsHook(GUARDRAIL_ID, GUARDRAIL_VERSION, ARC_POLICY_ARN)],
    tools=[retrieve],
    system_prompt=agent_instructions
)


# ----------------------------



# Session cache: session_id -> Agent (preserves conversation history across turns)
_SESSION_AGENTS: dict[str, Agent] = {}


@app.entrypoint
async def invoke(payload, context):
    """Handle an agent invocation from AgentCore Runtime."""
    prompt = payload.get("prompt", "")
    session_id = context.session_id
    logger.info("Received prompt (session=%s): %s", session_id, prompt[:80])

    if session_id and session_id in _SESSION_AGENTS:
        agent = _SESSION_AGENTS[session_id]
    else:
        # agent = Agent(model=_MODEL, tools=_TOOLS, system_prompt=SYSTEM_PROMPT)
        agent = myagent
        if session_id:
            _SESSION_AGENTS[session_id] = agent

    parts = []
    async for event in agent.stream_async(prompt):
        if "data" in event:
            parts.append(str(event["data"]))
    response = "".join(parts)
    # Strip inline <thinking>...</thinking> blocks so spans contain only the final answer
    response = re.sub(
        r"<thinking>.*?</thinking>", "", response, flags=re.DOTALL
    ).strip()
    return response


if __name__ == "__main__":
    app.run()

Overwriting agents/claimvalidation-agent-noAtoA.py


In [29]:
# For the created file, replace the placeholder values with environmental data
def replace_text_in_file(file_path, old_string, new_string):
    # Read the file content
    with open(file_path, 'r') as file:
        file_contents = file.read()

    # Perform the replacement(s)
    # The string replace() method returns a *new* string
    updated_contents = file_contents.replace(old_string, new_string)

    # Write the modified content back to the same file
    with open(file_path, 'w') as file:
        file.write(updated_contents)

    print(f"Text replacement completed successfully in {file_path}.")

# Example usage:
# Assuming a file named 'example.txt' exists with content "This is an old house."
replace_text_in_file('agents/claimvalidation-agent-noAtoA.py', '<REPLACE_BEDROCK_REGION>', BEDROCK_REGION)
replace_text_in_file('agents/claimvalidation-agent-noAtoA.py', '<REPLACE_ARC_POLICY_ARN>', ARC_POLICY_ARN)
replace_text_in_file('agents/claimvalidation-agent-noAtoA.py', '<REPLACE_GUARDRAIL_ID>', GUARDRAIL_ID)
replace_text_in_file('agents/claimvalidation-agent-noAtoA.py', '<REPLACE_GUARDRAIL_VERSION>', GUARDRAIL_VERSION)
replace_text_in_file('agents/claimvalidation-agent-noAtoA.py', '<REPLACE_KNOWLEDGE_BASE_ID>', KNOWLEDGE_BASE_ID)

Text replacement completed successfully in agents/claimvalidation-agent-noAtoA.py.
Text replacement completed successfully in agents/claimvalidation-agent-noAtoA.py.
Text replacement completed successfully in agents/claimvalidation-agent-noAtoA.py.
Text replacement completed successfully in agents/claimvalidation-agent-noAtoA.py.
Text replacement completed successfully in agents/claimvalidation-agent-noAtoA.py.


In [30]:
%%writefile agents/requirements.txt
boto3
botocore
bedrock-agentcore
bedrock-agentcore-starter-toolkit
strands-agents
strands-agents[a2a]
strands-agents-tools
pyyaml
ddgs
PyYAML
PyPDF2
opentelemetry-sdk
opentelemetry-exporter-otlp
opentelemetry-instrumentation-boto
opentelemetry-sdk-extension-aws
pydantic 
httpx
fastapi
uvicorn[standard]
requests

Overwriting agents/requirements.txt


In [31]:
from bedrock_agentcore_starter_toolkit import Runtime

_REGION = REGION or "us-east-1"

agentcore_runtime = Runtime()
agentcore_runtime.configure(
    entrypoint="agents/claimvalidation-agent-noAtoA.py",
    agent_name="myclaimvalidation_agent",
    region=_REGION,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    non_interactive=True,
)
print("Configuration complete.")

print("\nDeploying Claim Valdiation Assistant Agent ...")
print("  This takes ~5 minutes on first run (image build + push + runtime creation).")
print()

_launch = agentcore_runtime.launch(auto_update_on_conflict=True)

print(f"\nLaunch complete.")
print(f"  agent_id  : {_launch.agent_id}")
print(f"  agent_arn : {_launch.agent_arn}")

Entrypoint parsed: file=/home/ec2-user/SageMaker/multi-agent-arc-example/agents/claimvalidation-agent-noAtoA.py, bedrock_agentcore_name=claimvalidation-agent-noAtoA
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: myclaimvalidation_agent
Memory disabled
Network mode: PUBLIC


⚠️ Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64', so local builds
won't work.
Please use default launch command which will do a remote cross-platform build using code build.For deployment other
options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

📄 Using existing Dockerfile: /home/ec2-user/SageMaker/multi-agent-arc-example/Dockerfile

Generated .dockerignore: /home/ec2-user/SageMaker/multi-agent-arc-example/.dockerignore
Keeping 'myclaimvalidation_agent' as default agent
Bedrock AgentCore configured: /home/ec2-user/SageMaker/multi-agent-arc-example/.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'myclaimvalidation_agent' to account 161615149547 (us-east-1)
Generated image tag: 20260423-224833-606
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: myclaimvalidation_agent


Configuration complete.

Deploying Claim Valdiation Assistant Agent ...
  This takes ~5 minutes on first run (image build + push + runtime creation).

✅ Reusing existing ECR repository: 161615149547.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-myclaimvalidation_agent


ECR repository available: 161615149547.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-myclaimvalidation_agent
Getting or creating execution role for agent: myclaimvalidation_agent
Using AWS region: us-east-1, account ID: 161615149547
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-f2404ea203
✅ Reusing existing execution role: arn:aws:iam::161615149547:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-f2404ea203
Execution role available: arn:aws:iam::161615149547:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-f2404ea203
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: myclaimvalidation_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-f2404ea203
Reusing existing CodeBuild execution role: arn:aws:iam::161615149547:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-f2404ea203
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: myclaimvalidation_agent/source.zip
Updated CodeBuild


Launch complete.
  agent_id  : myclaimvalidation_agent-qixKSjEH56
  agent_arn : arn:aws:bedrock-agentcore:us-east-1:161615149547:runtime/myclaimvalidation_agent-qixKSjEH56


In [32]:
import time

print("Waiting for agent to reach READY status ...")

_POLL_INTERVAL = 15   # seconds between status checks
_MAX_WAIT      = 600  # 10-minute timeout

_elapsed = 0
while _elapsed < _MAX_WAIT:
    _status_result = agentcore_runtime.status()
    _agent_info    = _status_result.agent or {}
    _agent_status  = _agent_info.get("status", "UNKNOWN")
    print(f"  [{_elapsed:>3}s] status = {_agent_status}")

    if _agent_status in ("READY", "ACTIVE"):
        print(f"\nAgent is {_agent_status}. Proceeding.")
        break
    if _agent_status in ("FAILED", "CREATE_FAILED", "UPDATE_FAILED"):
        raise RuntimeError(
            f"Agent deployment failed with status '{_agent_status}'.\n"
            f"Details: {_agent_info}"
        )

    time.sleep(_POLL_INTERVAL)
    _elapsed += _POLL_INTERVAL
else:
    raise TimeoutError(
        f"Agent did not reach READY status within {_MAX_WAIT}s. "
        "Check the AgentCore console for details."
    )

Retrieved Bedrock AgentCore status for: myclaimvalidation_agent


Waiting for agent to reach READY status ...
  [  0s] status = READY

Agent is READY. Proceeding.


In [33]:
AGENT_ID     = _launch.agent_id
AGENT_ARN    = _launch.agent_arn
CW_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{AGENT_ID}-DEFAULT"

agentcore_client = boto3.client("bedrock-agentcore", region_name=_REGION)

print(f"AGENT_ID     : {AGENT_ID}")
print(f"AGENT_ARN    : {AGENT_ARN}")
print(f"CW_LOG_GROUP : {CW_LOG_GROUP}")

AGENT_ID     : myclaimvalidation_agent-qixKSjEH56
AGENT_ARN    : arn:aws:bedrock-agentcore:us-east-1:161615149547:runtime/myclaimvalidation_agent-qixKSjEH56
CW_LOG_GROUP : /aws/bedrock-agentcore/runtimes/myclaimvalidation_agent-qixKSjEH56-DEFAULT


In [34]:
# Persist agent info
%store AGENT_ID
%store AGENT_ARN
%store CW_LOG_GROUP
%store REGION

Stored 'AGENT_ID' (str)
Stored 'AGENT_ARN' (str)
Stored 'CW_LOG_GROUP' (str)
Stored 'REGION' (str)


## Step 3: Invoke the Agent to Generate Sessions

Before we can evaluate, we need agent sessions with CloudWatch spans. We'll invoke the agent
for several scenarios and record the session IDs for use with `EvaluationClient`.

Each session corresponds to one evaluation scenario.

In [40]:
def invoke_agent(prompt: str, session_id: str) -> str:
    """Send a single prompt to the Claim Validation assistant and return its text response."""
    resp = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode("utf-8"),
    )
    raw = resp["response"].read().decode("utf-8")
    parts = []
    for line in raw.splitlines():
        if line.startswith("data: "):
            chunk = line[len("data: "):]
            try:
                chunk = json.loads(chunk)
            except Exception:
                pass
            parts.append(str(chunk))
    return "".join(parts) if parts else raw


def run_session(turns: list[str], session_prefix: str) -> str:
    """Invoke a multi-turn session and return its session ID."""
    session_id = f"{session_prefix}-{uuid.uuid4()}"
    print(f"Session: {session_id}")
    for turn_input in turns:
        print(f"  > {turn_input[:100]}")
        response = invoke_agent(turn_input, session_id)
        print(f"  < {response[:1000]}")
    return session_id

In [41]:
# --- Single-turn sessions ---

print("=== Single-Turn Sessions ===")

session_claimLate = run_session(
    ["I was delivering goods for my company using my personal car. I got into an accident, this was over 3 months ago. Does my policy cover this?"],
    "claim-delivery-late"
)

# session_pto_balance = run_session(
#     ["What is the current PTO balance for employee EMP-001?"],
#     "pto-balance-check"
# )

# session_submit_pto = run_session(
#     ["Please submit a PTO request for employee EMP-001 from 2026-04-14 to 2026-04-16 for a family vacation."],
#     "submit-pto-request"
# )

# session_pay_stub = run_session(
#     ["Can you pull up the January 2026 pay stub for employee EMP-001?"],
#     "pay-stub-lookup"
# )

print("\nSingle-turn sessions created.")

=== Single-Turn Sessions ===
Session: claim-delivery-late-d65c6013-60c5-4333-92ec-ec14c45cc348
  > I was delivering goods for my company using my personal car. I got into an accident, this was over 3
  < "I'll help you determine if your personal auto insurance policy covers commercial use of your vehicle. Let me search for the relevant policy information regarding commercial use and coverage limitations.Based on my analysis of your auto insurance policy, I have determined the validity of your claim:\n\n## **This Claim is Invalid and no appraisal nor settlement is required**\n\n### Reason for Invalidity:\n\nYour claim is invalid due to a **commercial use exclusion** in your personal auto insurance policy. The policy explicitly excludes coverage for vehicles used \"for retail or wholesale delivery, including, but not limited to, the pickup, transport or delivery of magazines, newspapers, mail or food.\"\n\nSince you were delivering goods for your company using your personal vehicle, this

In [37]:
# # --- Multi-turn session: PTO planning ---

# print("=== Multi-Turn Session: PTO Planning ===")

# session_pto_planning = run_session(
#     [
#         "How many PTO days do I have left? My employee ID is EMP-001.",
#         "Great. I'd like to take December 23 to December 25 off. Please submit a request.",
#         "Remind me — what is the policy on rolling over unused PTO?",
#     ],
#     "pto-planning-session"
# )

# print("\nMulti-turn session created.")

=== Multi-Turn Session: PTO Planning ===
Session: pto-planning-session-2636acbf-8163-4b57-9259-9cddffd60164
  > How many PTO days do I have left? My employee ID is EMP-001.
  < "I don't have access to tools that can retrieve employee PTO (Paid Time Off) information or HR syste
  > Great. I'd like to take December 23 to December 25 off. Please submit 
  < "I don't have the ability to submit PTO requests or access HR systems to process time-off requests. 
  > Remind me — what is the policy on rolling over unused PTO?
  < "I can help you search for information about PTO rollover policies. Let me look for that information

Multi-turn session created.


In [ ]:
# # --- Multi-turn session: New employee onboarding ---

# print("=== Multi-Turn Session: New Employee Onboarding ===")

# session_onboarding = run_session(
#     [
#         "I just joined the company. What is the remote work policy?",
#         "How much PTO do I get as a new employee?",
#         "What life insurance benefit does the company provide?",
#         "Can you check the current PTO balance for employee EMP-042?",
#     ],
#     "new-employee-onboarding"
# )

# print("\nAll sessions created. Waiting 60s for CloudWatch log ingestion...")
# time.sleep(60)
# print("Ready to evaluate.")

## Step 5: EvaluationClient — Evaluate Existing Sessions

`EvaluationClient` is the right tool when you **already have agent sessions** logged in CloudWatch and you want to test them against your ground truth in a ad-hoc manner.
It looks up the agent's spans for a given `session_id` and runs evaluators against them. For these evaluations, you can pass in an expected response, assertions and expected trajectory. You can use the Built-in evaluators as well as the custom evaluators.

### Ground-Truth Reference Inputs

`ReferenceInputs` lets you supply optional ground truth:

| Field | Evaluators that use it | Description |
|---|---|---|
| `expected_response` | `Builtin.Correctness` | The ideal response text |
| `expected_trajectory` | `Builtin.TrajectoryExactOrderMatch`, `Builtin.TrajectoryInOrderMatch`, `Builtin.TrajectoryAnyOrderMatch` | Ordered list of tool names |
| `assertions` | `Builtin.GoalSuccessRate` | Free-text assertions the session should satisfy |

Evaluators that don't need ground truth (`Helpfulness`, `ResponseRelevance`) can be included in the same call.
Each evaluator only reads the fields it needs.

## Create Custom (LLM-as-a-Judge) Evaluators

In addition to built-in evaluators, you can define your own evaluation criteria using
**LLM-as-a-Judge custom evaluators**. These accept natural language instructions that
can reference **ground truth placeholders** automatically substituted at evaluation time.

### Ground truth placeholders

| Level | Available placeholders |
|---|---|
| **TRACE** | `{context}`, `{assistant_turn}`, `{expected_response}` |
| **SESSION** | `{context}`, `{available_tools}`, `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{assertions}` |

For example, a trace-level evaluator comparing response similarity would include
`{assistant_turn}` and `{expected_response}` in its instructions. When the evaluator runs,
the service substitutes those placeholders with the actual agent output and the
`expectedResponse` from `ReferenceInputs`.

### What we'll create

| Evaluator | Level | Placeholders | Description |
|---|---|---|---|
| `HRResponseSimilarity` | TRACE | `{assistant_turn}`, `{expected_response}` | How closely the agent's response matches the expected answer |
| `HRAssertionChecker` | SESSION | `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{assertions}` | Whether the agent called the right tools and satisfied all session assertions |

In [42]:
import uuid

_SUFFIX = uuid.uuid4().hex[:8]
_cp = boto3.client("bedrock-agentcore-control", region_name=_REGION)

# ---------------------------------------------------------------------------
# Trace-level: HRResponseSimilarity
# Compares the agent's response to the expected_response reference input.
# {assistant_turn} → actual agent output
# {expected_response} → expectedResponse field in ReferenceInputs
# ---------------------------------------------------------------------------
print("Creating HRResponseSimilarity (TRACE) ...")
_resp_sim = _cp.create_evaluator(
    evaluatorName=f"HRResponseSimilarity_{_SUFFIX}",
    level="TRACE",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "Compare the agent's response with the expected response.\n"
                "Agent response: {assistant_turn}\n"
                "Expected response: {expected_response}\n\n"
                "Rate how closely the agent's response matches the expected response. "
                "Focus on whether the key facts, numbers, and conclusions agree."
            ),
            "ratingScale": {
                "numerical": [
                    {
                        "value": 0.0,
                        "label": "not_similar",
                        "definition": "Response is factually different or missing key information from the expected response.",
                    },
                    {
                        "value": 0.5,
                        "label": "partially_similar",
                        "definition": "Response captures some expected content but omits or misrepresents parts.",
                    },
                    {
                        "value": 1.0,
                        "label": "highly_similar",
                        "definition": "Response is semantically equivalent to the expected response — all key facts match.",
                    },
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": "us.amazon.nova-lite-v1:0",
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_RESPONSE_SIMILARITY_ID = _resp_sim["evaluatorId"]
print(f"  evaluatorId : {CUSTOM_RESPONSE_SIMILARITY_ID}")

# ---------------------------------------------------------------------------
# Session-level: HRAssertionChecker
# Evaluates tool trajectory compliance and assertion satisfaction.
# {actual_tool_trajectory}   → tools the agent actually called
# {expected_tool_trajectory} → expectedTrajectory from ReferenceInputs
# {assertions}               → assertions list from ReferenceInputs
# ---------------------------------------------------------------------------
print("\nCreating HRAssertionChecker (SESSION) ...")
_assert_chk = _cp.create_evaluator(
    evaluatorName=f"HRAssertionChecker_{_SUFFIX}",
    level="SESSION",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "Evaluate whether the agent fulfilled the session requirements.\n\n"
                "Expected tool trajectory: {expected_tool_trajectory}\n"
                "Actual tool trajectory: {actual_tool_trajectory}\n"
                "Assertions to verify: {assertions}\n\n"
                "Score the agent on how well it followed the expected tool trajectory "
                "and satisfied every listed assertion."
            ),
            "ratingScale": {
                "numerical": [
                    {
                        "value": 0.0,
                        "label": "failed",
                        "definition": "Agent did not follow the trajectory and failed most assertions.",
                    },
                    {
                        "value": 0.5,
                        "label": "partial",
                        "definition": "Agent partially followed the trajectory or satisfied only some assertions.",
                    },
                    {
                        "value": 1.0,
                        "label": "passed",
                        "definition": "Agent followed the expected trajectory and satisfied all assertions.",
                    },
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": "us.amazon.nova-lite-v1:0",
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_ASSERTION_CHECKER_ID = _assert_chk["evaluatorId"]
print(f"  evaluatorId : {CUSTOM_ASSERTION_CHECKER_ID}")

print(f"\nCustom evaluators ready:")
print(f"  HRResponseSimilarity (TRACE)   : {CUSTOM_RESPONSE_SIMILARITY_ID}")
print(f"  HRAssertionChecker   (SESSION) : {CUSTOM_ASSERTION_CHECKER_ID}")

Creating HRResponseSimilarity (TRACE) ...
  evaluatorId : HRResponseSimilarity_3191b033-SZ0tMSC08F

Creating HRAssertionChecker (SESSION) ...
  evaluatorId : HRAssertionChecker_3191b033-CpT1aP5k1h

Custom evaluators ready:
  HRResponseSimilarity (TRACE)   : HRResponseSimilarity_3191b033-SZ0tMSC08F
  HRAssertionChecker   (SESSION) : HRAssertionChecker_3191b033-CpT1aP5k1h


In [43]:
from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs

eval_client = EvaluationClient(region_name=REGION)

print(f"EvaluationClient initialised (region={REGION})")
print(f"  {CUSTOM_RESPONSE_SIMILARITY_ID} → TRACE  (custom: HRResponseSimilarity)")
print(f"  {CUSTOM_ASSERTION_CHECKER_ID} → SESSION (custom: HRAssertionChecker)")

EvaluationClient initialised (region=us-east-1)
  HRResponseSimilarity_3191b033-SZ0tMSC08F → TRACE  (custom: HRResponseSimilarity)
  HRAssertionChecker_3191b033-CpT1aP5k1h → SESSION (custom: HRAssertionChecker)


In [44]:
# Helper function for printing
def display_eval_results(label: str, results: list) -> None:
    """Pretty-print EvaluationClient results as a markdown table."""
    rows = ["| Evaluator | Value | Label | Explanation |",
            "|---|---|---|---|"]
    for r in results:
        evaluator = r.get("evaluatorId", "")[:40]
        value = str(r.get("value", r.get("score", "N/A")))
        lbl = str(r.get("label", r.get("rating", "")))
        explanation = (r.get("explanation", r.get("reason", "")) or "")[:120].replace("\n", " ")
        error_code = r.get("errorCode")
        if error_code:
            lbl = f"ERR:{error_code}"
            explanation = (r.get("errorMessage", "") or "")[:120]
        rows.append(f"| `{evaluator}` | {value} | {lbl} | {explanation} |")

    if len(rows) == 2:  # only header rows, no data
        rows.append("| No results — session may be too recent or spans not yet visible | | | |")

    md = f"### {label}\n\n" + "\n".join(rows)
    display(Markdown(md))

### 5a. Single-Turn: PTO Balance — Correctness + Helpfulness + Custom ResponseSimilarity

We evaluate the PTO balance response against a known expected answer using `Builtin.Correctness`
and the custom `HRResponseSimilarity` evaluator (which uses the `{assistant_turn}` and
`{expected_response}` placeholders). Both measure factual accuracy but use different scoring rubrics.

In [45]:
groundtruth_response = f"""## **This Claim is Invalid and no appraisal nor settlement is required**

### Explanation for Invalid Claim:

Your auto insurance claim is **INVALID** due to a **clear policy exclusion** for commercial/business use of your personal vehicle. According to your policy documents, coverage specifically **does not apply** when your vehicle is being used "for retail or wholesale delivery, including, but not limited to, the pickup, transport or delivery of magazines, newspapers, mail or food."

Since you were "delivering goods for your company," this falls directly under the excluded commercial delivery activities outlined in your policy.

### Claims Policy Validation Explainability

## Finding 
**Finding Type:** Coverage Exclusion Violation

### Translation:
#### Premises:
- Vehicle was used for delivery of goods for company (commercial delivery activity)
- Policy excludes coverage for "retail or wholesale delivery" activities
- Accident occurred over 3 months ago (90+ days)

#### Claims:
- Commercial delivery use triggers policy exclusion
- No coverage applies under liability, medical payments, or comprehensive/collision parts of policy
- Late reporting (90+ days) compounds the coverage denial

#### Untranslated Claims:
- "Does my policy cover this?" - Answer: No, due to commercial use exclusion

**Confidence Score:** 95%

### Claims True Scenario:
The policy explicitly excludes coverage when a personal vehicle is used for commercial delivery purposes. This exclusion applies across all coverage parts (liability, medical payments, comprehensive/collision). Your use of the personal vehicle to deliver goods for your company constitutes commercial delivery activity, which is specifically excluded under the policy terms.
"""
print (groundtruth_response)

## **This Claim is Invalid and no appraisal nor settlement is required**

### Explanation for Invalid Claim:

Your auto insurance claim is **INVALID** due to a **clear policy exclusion** for commercial/business use of your personal vehicle. According to your policy documents, coverage specifically **does not apply** when your vehicle is being used "for retail or wholesale delivery, including, but not limited to, the pickup, transport or delivery of magazines, newspapers, mail or food."

Since you were "delivering goods for your company," this falls directly under the excluded commercial delivery activities outlined in your policy.

### Claims Policy Validation Explainability

## Finding 
**Finding Type:** Coverage Exclusion Violation

### Translation:
#### Premises:
- Vehicle was used for delivery of goods for company (commercial delivery activity)
- Policy excludes coverage for "retail or wholesale delivery" activities
- Accident occurred over 3 months ago (90+ days)

#### Claims:
- C

In [47]:
claimLate_results = eval_client.run(
    evaluator_ids=[
        "Builtin.Correctness",           # TRACE: compares with provided expected response
        "Builtin.Helpfulness",           # TRACE: no ground truth needed
        "Builtin.ResponseRelevance",     # TRACE: no ground truth needed
        CUSTOM_RESPONSE_SIMILARITY_ID,   # TRACE: custom — uses {assistant_turn} + {expected_response}
    ],
    session_id=session_claimLate,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_response = groundtruth_response,
    ),
)

display_eval_results("claim Late Results — Correctness + Quality + Custom ResponseSimilarity", claimLate_results)

### claim Late Results — Correctness + Quality + Custom ResponseSimilarity

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 1.0 | Correct | Both responses correctly identify that the claim is INVALID due to commercial use exclusion. The agent response accurate |
| `Builtin.Helpfulness` | 0.83 | Very Helpful | The user's goal is to understand whether their personal auto insurance policy covers an accident that occurred while del |
| `Builtin.ResponseRelevance` | 1.0 | Completely Yes | The user asked whether their personal auto insurance policy covers an accident that occurred over 3 months ago while del |
| `HRResponseSimilarity_3191b033-SZ0tMSC08F` | 0.9 |  | The agent's response closely matches the expected response. Both responses highlight the invalidity of the claim due to  |

In [ ]:
pto_balance_results = eval_client.run(
    evaluator_ids=[
        "Builtin.Correctness",           # TRACE: compares with provided expected response
        "Builtin.Helpfulness",           # TRACE: no ground truth needed
        "Builtin.ResponseRelevance",     # TRACE: no ground truth needed
        CUSTOM_RESPONSE_SIMILARITY_ID,   # TRACE: custom — uses {assistant_turn} + {expected_response}
    ],
    session_id=session_pto_balance,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_response="Employee EMP-001 has 10 remaining PTO days out of 15 total (5 days used).",
    ),
)

display_eval_results("PTO Balance — Correctness + Quality + Custom ResponseSimilarity", pto_balance_results)

### 5b. Single-Turn: PTO Submission — Assertions + Trajectory + Custom AssertionChecker

This cell runs both built-in trajectory evaluators **and** the custom `HRAssertionChecker`
(which uses `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, and `{assertions}` placeholders)
plus the custom `HRResponseSimilarity` for the response. This lets you compare built-in vs. custom
scoring side by side.

In [ ]:
submit_pto_results = eval_client.run(
    evaluator_ids=[
        "Builtin.GoalSuccessRate",           # SESSION: built-in assertion evaluator
        "Builtin.TrajectoryExactOrderMatch", # SESSION: built-in trajectory evaluator
        "Builtin.TrajectoryAnyOrderMatch",   # SESSION: built-in trajectory evaluator
        "Builtin.Correctness",               # TRACE: built-in response accuracy
        CUSTOM_RESPONSE_SIMILARITY_ID,       # TRACE (custom): {assistant_turn} + {expected_response}
    ],
    session_id=session_submit_pto,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_trajectory=["submit_pto_request"],
        assertions=[
            "Agent called submit_pto_request for employee EMP-001",
            "Agent confirmed the PTO request was approved",
            "Agent provided a request ID (e.g. PTO-2026-001)",
        ],
        expected_response="PTO request submitted and approved for EMP-001 from 2026-04-14 to 2026-04-16.",
    ),
)

display_eval_results("PTO Submission — Built-in + Custom ResponseSimilarity", submit_pto_results)

### 5c. Single-Turn: Pay Stub — Factual Correctness

Factual data retrieval scenarios are well-suited for `Builtin.Correctness` combined with
`Builtin.GoalSuccessRate`. The expected_response provides the ground truth figures.

In [ ]:
pay_stub_results = eval_client.run(
    evaluator_ids=[
        "Builtin.Correctness",
        "Builtin.GoalSuccessRate",
    ],
    session_id=session_pay_stub,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_response="EMP-001 January 2026: gross pay $8,333.33, net pay $5,362.50.",
        assertions=[
            "Agent called get_pay_stub for EMP-001 period 2026-01",
            "Agent reported the correct gross pay of $8,333.33",
            "Agent reported the correct net pay of $5,362.50",
        ],
    ),
)

display_eval_results("Pay Stub Lookup — Correctness + GoalSuccessRate", pay_stub_results)

### 5d. Multi-Turn: PTO Planning Session (3 turns) + Custom AssertionChecker

For multi-turn sessions, `EvaluationClient` fetches all spans for the session and evaluates
the complete conversation. The trajectory and assertions apply across all turns.

This scenario also exercises the custom `HRAssertionChecker` evaluator (SESSION level),
which uses `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, and `{assertions}`
placeholders. A 3-turn session with distinct tool calls per turn gives the evaluator
a rich trajectory to compare against the expected sequence.

In [ ]:
pto_planning_results = eval_client.run(
    evaluator_ids=[
        "Builtin.GoalSuccessRate",
        "Builtin.TrajectoryExactOrderMatch",
        "Builtin.TrajectoryInOrderMatch",
        "Builtin.TrajectoryAnyOrderMatch",
        "Builtin.Helpfulness",
        CUSTOM_ASSERTION_CHECKER_ID,   # SESSION (custom): {actual_tool_trajectory} + {expected_tool_trajectory} + {assertions}
    ],
    session_id=session_pto_planning,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_trajectory=["get_pto_balance", "submit_pto_request", "lookup_hr_policy"],
        assertions=[
            "Agent correctly reported 10 remaining PTO days for EMP-001 in turn 1",
            "Agent submitted a PTO request for December 23-25, 2026 in turn 2",
            "Agent correctly stated the 5-day PTO rollover limit in turn 3",
        ],
    ),
)

display_eval_results("PTO Planning — Multi-Turn (3 turns) + Custom AssertionChecker", pto_planning_results)

## Step 6: OnDemandEvaluationDatasetRunner — Automated Dataset Evaluation

`OnDemandEvaluationDatasetRunner` is the right tool when you have a **test dataset** and want to:
1. Automatically invoke your agent for each scenario
2. Collect CloudWatch spans
3. Run evaluators against each scenario's results

This is ideal for regression testing, CI/CD pipelines, and batch evaluation against curated datasets.

### Dataset structure

A dataset consists of **scenarios**, each with one or more **turns**. Optional ground-truth fields:
- `Turn.expected_response` — per-turn expected answer
- `PreDefinedScenario.expected_trajectory` — ordered list of tool names
- `PreDefinedScenario.assertions` — session-level assertions

### How OnDemandEvaluationDatasetRunner works

```
For each scenario:
  1. Create a new session ID
  2. Call your agent_invoker function for each turn
  3. Wait for CloudWatch spans to appear (evaluation_delay_seconds)
  4. Submit spans + ground truth to the evaluation service
  5. Collect and return results
```

In [ ]:
from bedrock_agentcore.evaluation import (
    AgentInvokerInput,
    AgentInvokerOutput,
    CloudWatchAgentSpanCollector,
    Dataset,
    EvaluationRunConfig,
    OnDemandEvaluationDatasetRunner,
    EvaluatorConfig,
    Turn,
    PredefinedScenario,
)

In [ ]:
def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """
    Called by OnDemandEvaluationDatasetRunner once per turn. Invoke the HR assistant
    and return the text response.

    AgentInvokerInput fields:
      - payload:    The turn input (str or dict) from the dataset.
      - session_id: Framework-managed session ID, stable across all turns
                    in a scenario. Pass it to your agent for conversation continuity.
    """
    payload = invoker_input.payload
    body = {"prompt": payload} if isinstance(payload, str) else payload

    resp = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=invoker_input.session_id,
        payload=json.dumps(body).encode("utf-8"),
    )

    raw = resp["response"].read().decode("utf-8")
    parts = []
    for line in raw.splitlines():
        if line.startswith("data: "):
            chunk = line[len("data: "):]
            try:
                chunk = json.loads(chunk)
            except Exception:
                pass
            parts.append(str(chunk))
    return AgentInvokerOutput(agent_output="".join(parts) if parts else raw)

### 6a. Define the Evaluation Dataset

We define scenarios inline. A mix of single-turn and multi-turn scenarios exercises
different aspects of the agent.

In [ ]:
dataset = Dataset(
    scenarios=[
        # --- Single-turn: PTO balance ---
        PredefinedScenario(
            scenario_id="pto-balance-check",
            turns=[
                Turn(
                    input="What is the current PTO balance for employee EMP-001?",
                    expected_response="Employee EMP-001 has 10 remaining PTO days out of 15 total (5 days used).",
                )
            ],
            expected_trajectory=["get_pto_balance"],
            assertions=[
                "Agent called get_pto_balance with employee_id=EMP-001",
                "Agent reported 10 remaining PTO days",
            ],
        ),

        # --- Single-turn: HR policy lookup ---
        PredefinedScenario(
            scenario_id="pto-policy-lookup",
            turns=[
                Turn(
                    input="What is the company PTO policy?",
                    expected_response="Full-time employees accrue 15 days of PTO per year. Requests must be submitted at least 2 business days in advance. Up to 5 unused days roll over each year.",
                )
            ],
            expected_trajectory=["lookup_hr_policy"],
            assertions=[
                "Agent called lookup_hr_policy with topic=pto",
                "Agent mentioned the 15-day annual accrual for full-time employees",
                "Agent mentioned the 2 business day advance notice requirement",
            ],
        ),

        # --- Single-turn: 401k benefits ---
        PredefinedScenario(
            scenario_id="401k-info",
            turns=[
                Turn(
                    input="How does the 401k match work?",
                    expected_response="The company matches 100% of contributions up to 4% of salary, plus 50% on the next 2%, for a total effective match of up to 5%. The match vests over 3 years.",
                )
            ],
            expected_trajectory=["get_benefits_summary"],
            assertions=[
                "Agent called get_benefits_summary with benefit_type=401k",
                "Agent correctly described the 4% full match and 50% match on next 2%",
                "Agent mentioned the 3-year vesting schedule",
            ],
        ),

        # --- Single-turn: check balance then submit PTO ---
        PredefinedScenario(
            scenario_id="check-and-submit-pto",
            turns=[
                Turn(
                    input="Check the PTO balance for EMP-002, and if they have at least 2 days, submit a request for 2026-05-26 to 2026-05-27.",
                    expected_response="EMP-002 has 3 remaining PTO days. PTO request submitted and approved for 2026-05-26 to 2026-05-27.",
                )
            ],
            expected_trajectory=["get_pto_balance", "submit_pto_request"],
            assertions=[
                "Agent first called get_pto_balance for EMP-002",
                "Agent confirmed 3 remaining days is sufficient",
                "Agent then called submit_pto_request for the correct dates",
            ],
        ),

        # --- Multi-turn: benefits exploration ---
        PredefinedScenario(
            scenario_id="benefits-exploration",
            turns=[
                Turn(
                    input="Can you walk me through the health insurance options?",
                    expected_response="The company covers 90% of premiums for employee-only coverage. Three plans are available: Blue Shield PPO, Kaiser HMO, and HDHP with HSA.",
                ),
                Turn(
                    input="What about dental?",
                    expected_response="The dental plan covers 100% of preventive care, 80% of basic restorative care, and 50% of major work, with a $2,000 annual maximum.",
                ),
                Turn(
                    input="And how much does the company contribute to the 401k?",
                    expected_response="The company matches 100% up to 4% of salary, plus 50% on the next 2%, for a total effective match of up to 5%.",
                ),
            ],
            expected_trajectory=["get_benefits_summary", "get_benefits_summary", "get_benefits_summary"],
            assertions=[
                "Agent called get_benefits_summary three times across the conversation",
                "Agent correctly described health, dental, and 401k benefits in their respective turns",
                "Agent maintained conversational context across all three turns",
            ],
        ),
    ]
)

print(f"Dataset contains {len(dataset.scenarios)} scenarios.")

### 6b. Configure and Run OnDemandEvaluationDatasetRunner

In [ ]:
# Span collector: polls CloudWatch for OTel spans emitted by the agent
span_collector = CloudWatchAgentSpanCollector(
    log_group_name=CW_LOG_GROUP,
    region=REGION,
    max_wait_seconds=180,
    poll_interval_seconds=15,
)

# Evaluator level cache — built-ins + custom evaluators
EVALUATOR_LEVELS = {
    "Builtin.GoalSuccessRate":            "SESSION",
    "Builtin.TrajectoryExactOrderMatch":  "SESSION",
    "Builtin.TrajectoryInOrderMatch":     "SESSION",
    "Builtin.TrajectoryAnyOrderMatch":    "SESSION",
    "Builtin.Correctness":                "TRACE",
}
# Custom evaluators (These are the custom evaluators we created for HR Response Similarity and HRAssertionChecker)
EVALUATOR_LEVELS[CUSTOM_RESPONSE_SIMILARITY_ID] = "TRACE"
EVALUATOR_LEVELS[CUSTOM_ASSERTION_CHECKER_ID]   = "SESSION"

# Evaluator configuration — mix of built-in and custom evaluators
config = EvaluationRunConfig(
    evaluator_config=EvaluatorConfig(
        evaluator_ids=[
            "Builtin.Correctness",               # TRACE — expected_response
            "Builtin.GoalSuccessRate",            # SESSION — assertions
            "Builtin.TrajectoryExactOrderMatch",  # SESSION — expected_trajectory
            "Builtin.TrajectoryInOrderMatch",     # SESSION — expected_trajectory
            "Builtin.TrajectoryAnyOrderMatch",    # SESSION — expected_trajectory 
            CUSTOM_RESPONSE_SIMILARITY_ID,       # TRACE (custom) — {assistant_turn} + {expected_response}
            CUSTOM_ASSERTION_CHECKER_ID,         # SESSION (custom) — {actual_tool_trajectory} + {assertions}
        ]
    ),
    evaluation_delay_seconds=180,
    max_concurrent_scenarios=3,
)

runner = OnDemandEvaluationDatasetRunner(region=REGION)
runner._evaluator_level_cache.update(EVALUATOR_LEVELS)

print("OnDemandEvaluationDatasetRunner configured. Starting evaluation...")
print(f"  Scenarios : {len(dataset.scenarios)}")
print(f"  Evaluators: {len(config.evaluator_config.evaluator_ids)} "
      f"(7 built-in + 2 custom)")
print(f"  Delay     : {config.evaluation_delay_seconds}s (waiting for CloudWatch ingestion)")

In [ ]:
# Run the evaluation.
# OnDemandEvaluationDatasetRunner will:
#   1. Invoke agent_invoker for each turn in each scenario
#   2. Wait evaluation_delay_seconds for CloudWatch ingestion
#   3. Submit spans to the evaluation service
#   4. Return aggregated results

eval_result = runner.run(
    config=config,
    dataset=dataset,
    agent_invoker=agent_invoker,
    span_collector=span_collector,
)

completed = sum(1 for sr in eval_result.scenario_results if sr.status == "COMPLETED")
failed = sum(1 for sr in eval_result.scenario_results if sr.status == "FAILED")
print(f"\nEvaluation complete: {completed} completed, {failed} failed out of {len(eval_result.scenario_results)} scenarios.")

### 6c. Inspect Results

In [ ]:
def display_runner_results(eval_result) -> None:
    """Display OnDemandEvaluationDatasetRunner results as a markdown table per scenario."""
    for sr in eval_result.scenario_results:
        if sr.status == "FAILED":
            display(Markdown(f"**Scenario `{sr.scenario_id}`** — FAILED: {sr.error}"))
            continue

        rows = ["| Evaluator | Value | Label | Explanation |",
                "|---|---|---|---|"]
        for er in sr.evaluator_results:
            for res in er.results:
                value = str(res.get("value", res.get("score", "N/A")))
                lbl = str(res.get("label", res.get("rating", "")))
                explanation = (res.get("explanation", "") or "")[:130].replace("\n", " ")
                error_code = res.get("errorCode")
                if error_code:
                    lbl = f"ERR:{error_code}"
                    explanation = (res.get("errorMessage", "") or "")[:130]
                rows.append(f"| `{er.evaluator_id[:40]}` | {value} | {lbl} | {explanation} |")

        md = f"### Scenario: `{sr.scenario_id}`\n\n" + "\n".join(rows)
        display(Markdown(md))


display_runner_results(eval_result)

In [ ]:
# Aggregate summary: average score per evaluator across all scenarios
from collections import defaultdict

scores_by_evaluator = defaultdict(list)
for sr in eval_result.scenario_results:
    if sr.status != "COMPLETED":
        continue
    for er in sr.evaluator_results:
        for res in er.results:
            if "value" in res and res["value"] is not None and not res.get("errorCode"):
                scores_by_evaluator[er.evaluator_id].append(float(res["value"]))

print("\nEvaluator Summary (average score across all scenarios)")
print("=" * 60)
for evaluator_id, scores in sorted(scores_by_evaluator.items()):
    avg = sum(scores) / len(scores)
    print(f"  {evaluator_id:<45} avg={avg:.2f}  (n={len(scores)})")

### 6d. Save Results to File

In [ ]:
import os
from datetime import datetime

os.makedirs("results", exist_ok=True)
timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
results_path = f"results/groundtruth_eval_{timestamp}.json"

with open(results_path, "w") as f:
    json.dump(eval_result.model_dump(), f, indent=2, default=str)

print(f"Results saved to: {results_path}")

## Step 7: Cleanup

Delete the agent runtime endpoint when you're done to avoid ongoing costs.

In [ ]:
# Uncomment to delete the agent runtime
# agent_runtime.delete()
# print("Agent runtime deleted.")

print("Cleanup skipped. Uncomment the cell above to delete the agent runtime.")

### Key takeaways

| | EvaluationClient | OnDemandEvaluationDatasetRunner |
|---|---|---|
| **When to use** | You have existing sessions | You have a test dataset |
| **Best for** | Post-hoc analysis, debugging | Regression testing, CI/CD |
| **Input** | session_id | Dataset of scenarios |

### Built-in evaluator reference

| Evaluator | Level | Ground truth required |
|---|---|---|
| `Builtin.Correctness` | TRACE | `expected_response` |
| `Builtin.GoalSuccessRate` | SESSION | `assertions` |
| `Builtin.TrajectoryExactOrderMatch` | SESSION | `expected_trajectory` |
| `Builtin.TrajectoryInOrderMatch` | SESSION | `expected_trajectory` |
| `Builtin.TrajectoryAnyOrderMatch` | SESSION | `expected_trajectory` |


### Custom evaluator ground truth placeholders

Custom (LLM-as-a-judge) evaluators reference ground truth via placeholders in their `instructions`.

| Level | Placeholder | Filled from |
|---|---|---|
| TRACE | `{assistant_turn}` | Agent's actual response |
| TRACE | `{expected_response}` | `ReferenceInputs.expected_response` |
| TRACE | `{context}` | Session context |
| SESSION | `{actual_tool_trajectory}` | Tools called by the agent |
| SESSION | `{expected_tool_trajectory}` | `ReferenceInputs.expected_trajectory` |
| SESSION | `{assertions}` | `ReferenceInputs.assertions` |
| SESSION | `{available_tools}` | Tools available to the agent |